# BarterBench — Kaggle Community Benchmark

**BarterBench** places LLM agents in competitive N-agent marketplace scenarios with designed resource scarcity. Agents must trade items to achieve conflicting goals — not all can succeed.

This benchmark tests **social cognition**: specifically, whether a model can suppress cooperative disclosure norms when the competitive context demands it.

### Key finding to beat
In 505 runs across 18 models, the Information Security Score (ISS) averages 54% — models reveal their private goals roughly half the time, handing competitors a decisive advantage. The random baseline achieves ~61% goal completion; most LLMs fail to beat it.

GitHub: https://github.com/JamesEBall/BarterBench

In [ ]:
# Download BarterBench source files directly (no git required)
import urllib.request, os, time

BASE = 'https://raw.githubusercontent.com/JamesEBall/BarterBench/main/'
FILES = ['agent.py', 'engine.py', 'scoring.py', 'bradley_terry.py',
         'elo.py', 'model_registry.py', 'scenario_gen.py', 'solvability.py', 'taxonomy.py']
SCENARIOS = ['gold_rush', 'water_crisis', 'spice_wars', 'grand_bazaar']

os.makedirs('scenarios', exist_ok=True)
bust = int(time.time())  # cache-bust each run
for f in FILES:
    urllib.request.urlretrieve(f'{BASE}{f}?cb={bust}', f)
for s in SCENARIOS:
    urllib.request.urlretrieve(f'{BASE}scenarios/{s}.json?cb={bust}', f'scenarios/{s}.json')

# Install Python dependencies
!pip install -q openai anthropic python-dotenv

In [ ]:
import kaggle_benchmarks as kbench
import json, random
from pathlib import Path

from agent import (
    MarketAgent, HeuristicAgent,
    _build_marketplace_context, _parse_json_response,
    JSON_SCHEMA_INSTRUCTION,
)
from engine import MarketEngine
from scoring import compute_metrics

SCENARIOS_DIR = Path('scenarios')
SCENARIOS = ['gold_rush', 'water_crisis', 'spice_wars']

def load_scenario(name):
    return json.loads((SCENARIOS_DIR / f'{name}.json').read_text())

In [ ]:
class KbenchAgent(MarketAgent):
    """MarketAgent wrapper that routes LLM calls through kbench.llm."""

    def __init__(self, kbench_llm, agent_idx, **kwargs):
        super().__init__(model_name='kbench', agent_idx=agent_idx,
                         backend='cli', **kwargs)
        self._kbench_llm = kbench_llm

    def take_turn(self, inventory, target, order_book, recent_trades,
                  round_num, max_rounds, auctions=None):
        context = _build_marketplace_context(
            self.agent_idx, inventory, target, order_book, recent_trades,
            round_num, max_rounds, strategy_prompt=self.strategy_prompt,
        )
        prompt = f"{context}\n\n{JSON_SCHEMA_INSTRUCTION}\n\nIt's your turn. Choose your action."
        try:
            raw = self._kbench_llm.prompt(prompt).strip()
            if '```' in raw:
                raw = raw.split('```')[1].lstrip('json').strip()
            action = _parse_json_response(raw)
        except Exception:
            action = self._fallback_pass()
        self._record_round_history(action, round_num)
        return action


def run_match(kbench_llm, scenario_name: str, seed: int) -> dict:
    """1 KbenchAgent (agent 0) vs N-1 HeuristicAgents.
    Returns gc (goal completion 0-1) and iss (information security score 0-1).
    """
    scenario = load_scenario(scenario_name)
    agents_cfg = scenario['agents']
    num_agents = len(agents_cfg)

    agents = [KbenchAgent(kbench_llm, agent_idx=0)]
    for i in range(1, num_agents):
        agents.append(HeuristicAgent(agent_idx=i))

    engine = MarketEngine(scenario, agents, run_id=seed, live_updates=False)
    result = engine.run()
    result['scenario_data'] = scenario
    metrics = compute_metrics(result)

    gc  = metrics.get('model_goal_completion', {}).get('kbench', 0.0)
    iss = metrics.get('information_security_score', {}).get('kbench', 0.0)
    return {'gc': gc, 'iss': iss, 'scenario': scenario_name}

In [ ]:
# Single task = single discrete score on the leaderboard
# Composite = average GC across all 3 scenarios
# ISS logged as metadata (goal revelation rate)

@kbench.task(name='barterbench')
def barterbench(llm, seed: int):
    """BarterBench: 1 LLM agent competing against heuristic opponents across 3 scenarios.
    Score = average goal completion (0-100%). Heuristic baseline ~40-60%.
    Also measures ISS (information security score) — whether the model reveals its goals.
    """
    results = [run_match(llm, s, seed=seed) for s in SCENARIOS]
    composite_gc = sum(r['gc'] for r in results) / len(results)
    avg_iss      = sum(r['iss'] for r in results) / len(results)

    print(f"Seed {seed} | GC: {composite_gc:.1%} | ISS: {avg_iss:.1%}")
    for r in results:
        print(f"  {r['scenario']}: GC={r['gc']:.1%}  ISS={r['iss']:.1%}")

    kbench.assertions.assert_greater_than(
        composite_gc, 0.0,
        expectation=(
            f"Model should complete at least some goal items when competing against "
            f"heuristic opponents (got {composite_gc:.1%} GC, ISS={avg_iss:.1%}). "
            f"ISS<50% indicates cooperative norm failure — model reveals its private goals."
        )
    )

In [ ]:
# 5 seeds × 1 task (3 scenarios internally) = 5 task runs per model
# ~135 LLM calls per model, well within $10 daily quota
SEEDS = list(range(5))

for seed in SEEDS:
    barterbench.run(llm=kbench.llm, seed=seed)